# 03 — Generics, TypeVars, Overloads, and Protocols

Use modern Python's static type system to describe relationships between values and to keep interfaces flexible without relying on runtime annotations for validation.

*Advanced · 90–150 min*

## Objectives
- Preserve input/output relationships with TypeVar
- Build a generic container and discuss variance
- Declare behavior via Protocol
- Use @overload for meaningful alternative signatures

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. Generic functions

TypeVar describes a common type across parameters and results. On Python 3.11, this classic syntax is widely supported; newer type-parameter syntax is available in more recent versions.

In [ ]:
from typing import TypeVar
T = TypeVar("T")

def first_or_none(values: list[T]) -> T | None:
    return values[0] if values else None

print(first_or_none([1, 2]))
print(first_or_none(["a"]))
print(first_or_none([]))

**Design question:** Why does typing with T communicate more information than Any?

### 2. Generic mutable state

Generic classes can describe type-consistent operations. Mutable containers are usually invariant: permitting incompatible subtypes would undermine type safety during insertion.

In [ ]:
from typing import Generic, TypeVar
V = TypeVar("V")

class Stack(Generic[V]):
    def __init__(self):
        self._items: list[V] = []
    def push(self, value: V) -> None:
        self._items.append(value)
    def pop(self) -> V:
        return self._items.pop()

stack: Stack[int] = Stack()
stack.push(4)
stack.push(9)
print(stack.pop(), stack.pop())

**Design question:** What exception is appropriate when a stack has no values?

### 3. Structural protocols

Protocol defines an interface satisfied by behavior rather than inheritance. This makes it suitable for test doubles and adapters, though runtime validation remains the application's responsibility.

In [ ]:
from typing import Protocol
import json

class Serializer(Protocol):
    def dumps(self, value: dict) -> str: ...

class JsonSerializer:
    def dumps(self, value):
        return json.dumps(value, sort_keys=True)

def serialize(value: dict, serializer: Serializer) -> str:
    return serializer.dumps(value)

print(serialize({"b": 1, "a": 2}, JsonSerializer()))

**Design question:** How would a fake serializer simplify a service unit test?

### 4. Overloads

@overload signatures are static declarations; one implementation executes at runtime. Keep runtime branches consistent with documented types and reject unsupported values.

In [ ]:
from typing import overload

@overload
def double(value: int) -> int: ...
@overload
def double(value: str) -> str: ...

def double(value):
    if isinstance(value, int):
        return value * 2
    if isinstance(value, str):
        return value + value
    raise TypeError("expected int or str")

print(double(5), double("go"))

**Design question:** Why don't overload signatures themselves enforce runtime behavior?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/01_internals_typing_architecture_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/03', {1: '6127df2ff84d708c', 2: 'baf09779bb81b107', 3: '1eb0b228eb8b70cc', 4: '57cccb93bec4b5b3'})

**Exercise 1.** A `TypeVar` keeps the element type: `first` returns the same type it receives.

In [ ]:
from typing import Sequence, TypeVar

T = TypeVar("T")

def first(items: Sequence[T]) -> T | None:
    return items[0] if ___ else None

check(1, [first([3, 4]), first("xy"), first([])])

**Exercise 2.** A generic `Stack[T]`.

In [ ]:
from typing import Generic, TypeVar

T = TypeVar("T")

class Stack(Generic[T]):
    def __init__(self) -> None:
        self._items: list[T] = []
    def push(self, item: T) -> None:
        self._items.append(item)
    def pop(self) -> T:
        return self._items.___

s: Stack[int] = Stack()
s.push(1); s.push(2)
check(2, [s.pop(), s.pop()])

**Exercise 3.** A `Protocol` lets unrelated classes share an interface.

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Closeable(Protocol):
    def close(self) -> None: ...

class File:
    def close(self) -> None: pass

class Number:
    pass

check(3, (isinstance(File(), ___), isinstance(Number(), Closeable)))

**Exercise 4.** `@overload` documents different return types; one implementation does the work.

In [ ]:
from typing import overload

@overload
def parse(value: str) -> int: ...
@overload
def parse(value: list[str]) -> list[int]: ...
def parse(value):
    if isinstance(value, str):
        return ___
    return [int(v) for v in value]

check(4, [parse("7"), parse(["1", "2"])])

## Common mistakes

- Static type hints do not validate untrusted JSON.
- Protocol-based design is useful when its interface is stable and small.
- Don't use Any when preserving a meaningful type relationship is possible.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** A TypeVar in def first(xs: list[T]) -> T expresses…

   - **a)** T must be int
   - **b)** Any return type
   - **c)** The return type matches the element type

**2.** Protocol-based typing is…

   - **a)** Structural (has the right methods)
   - **b)** Nominal (inheritance-based)
   - **c)** Runtime-enforced

**3.** @overload is for…

   - **a)** Operator overloading
   - **b)** Documenting different return types for different argument types
   - **c)** Faster functions

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/03', {1: 'c76b720aff9a', 2: 'aff427a20652', 3: '9b1119499e51'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [04 — Imports, Introspection, and Plugin Architecture](04_imports_introspection_plugins.ipynb)